In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from matplotlib.colors import ListedColormap
from ppu.generator import Circular, GaussianBlobs, Moons, RingBlobs
from ppu.methods.mlp import MLP
from ppu.methods.tracin import get_tracin
from ppu.methods.utils import get_dataset, get_models
from ppu.viz.tracin_plot import draw
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.inspection._plot.decision_boundary import _check_boundary_response_method


In [ ]:
classifier = MLP(hidden_channels=[30, 100, 50, 1], patience=30, frequency=3)

n_ticks = 100
n_samples = 2000  #train instances


generator = {
    "GaussianBlobs" : GaussianBlobs,
    "Circular" : Circular,
    "Moons" : Moons,
    "RingBlobs" : RingBlobs
}

In [ ]:
gen = Circular
seed = 1

data = get_dataset(seed, gen, n_samples=n_samples)
nn  = get_models(classifier, gen, reps=1, n_samples=n_samples, seeds=[seed])[0]

(X_train, y_train), (X_test, y_test) = data
eps = 1.
x_min, x_max = X_train[:, 0].min() - eps, X_train[:, 0].max() + eps
y_min, y_max = X_train[:, 1].min() - eps, X_train[:, 1].max() + eps

x = np.linspace(
        x_min, x_max, n_ticks
    )  # for the heatmap we need to creat the grid first, so n_ticks are the density of the grid nodes
y = np.linspace(y_min, y_max, n_ticks)
xs, ys = np.meshgrid(x, y)
X_grid = np.c_[xs.ravel(), ys.ravel()]


In [ ]:
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=ListedColormap(["#FF0000", "#0000FF"]), edgecolors="k")
plt.xlim(x_min, x_max)  # axis range
plt.ylim(y_min, y_max)
plt.show()

In [ ]:
response = _check_boundary_response_method(nn, "auto")(
    X_grid
)  # the function returns the probability of points(inputs) belong to each class

if len(response.shape) != 1:
    response = response[
        :, 1
    ]  # since there's only 2 classes so 1 can represent the other(the prob sums up to 1)


cm = plt.cm.RdBu
display = DecisionBoundaryDisplay(
    xx0=xs, xx1=ys, response=response.reshape(xs.shape)
)  # class to draw DecisionBoundary
display.plot(cmap=cm, alpha=0.8)

In [ ]:
loss_t_1, loss_t_2, tracin_t = get_tracin(X_train, y_train, nn, train=(X_train, y_train))

In [ ]:
import pandas as pd

In [ ]:
X_t_df = pd.DataFrame(X_train, columns=["x", "y"])
X_t_df["tracein"] = tracin_t

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.tricontourf(X_train[:, 0], X_train[:, 1], tracin_t)
ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], alpha=0.3)
ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], alpha=0.3)

In [ ]:

sns.kdeplot(X_t_df, x="x", y="y", ax=ax, palette="viridis")
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1])
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1])


In [ ]:
plt.hist(tracin_t, bins='auto', log=True)
plt.show()

In [ ]:
import statsmodels.api as sm

ecdf = sm.distributions.ECDF(tracin_t)

xt_tracin = np.linspace(min(tracin_t), max(tracin_t))
yt_tracin = ecdf(xt_tracin)
plt.step(xt_tracin, yt_tracin)
plt.show()

In [ ]:
gen = RingBlobs
seed = 1

data = get_dataset(seed, gen, n_samples=n_samples, class_sep=8.0)
nn  = get_models(classifier, gen, reps=1, n_samples=n_samples, seeds=[seed], class_sep=8.0)[0]

(X_train, y_train), (X_test, y_test) = data
eps = 1.
x_min, x_max = X_train[:, 0].min() - eps, X_train[:, 0].max() + eps
y_min, y_max = X_train[:, 1].min() - eps, X_train[:, 1].max() + eps

x = np.linspace(
        x_min, x_max, n_ticks
    )  # for the heatmap we need to creat the grid first, so n_ticks are the density of the grid nodes
y = np.linspace(y_min, y_max, n_ticks)
xs, ys = np.meshgrid(x, y)
X_grid = np.c_[xs.ravel(), ys.ravel()]

cm = plt.cm.RdBu

In [ ]:
from ppu.viz.tracin_plot import tracin_plot_contour

itrs = [i+1 for i in range(5)]
figure = tracin_plot_contour(Circular, itrs, 2000, classifier, eps=10.)

In [ ]:
cm_bright = ListedColormap(["#FF0000", "#0000FF"])  # color for data points#
# Plot the training points
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright, edgecolors="k")

In [ ]:
classifier = MLP(hidden_channels=[30, 100, 200, 100, 50, 1], patience=10, frequency=9, lr=1e-2, device="mps")

In [ ]:
itr = 3
resample = False
gen = RingBlobs
seed = 2

data = get_dataset(seed, gen, n_samples=n_samples, class_sep=13.)
nn  = get_models(classifier, gen, reps=1, n_samples=n_samples, seeds=[seed], class_sep=13.)[0]

(X_train, y_train), (X_test, y_test) = data
eps = 1.
x_min, x_max = X_train[:, 0].min() - eps, X_train[:, 0].max() + eps
y_min, y_max = X_train[:, 1].min() - eps, X_train[:, 1].max() + eps

x = np.linspace(
        x_min, x_max, n_ticks
    )  # for the heatmap we need to creat the grid first, so n_ticks are the density of the grid nodes
y = np.linspace(y_min, y_max, n_ticks)
xs, ys = np.meshgrid(x, y)
X_grid = np.c_[xs.ravel(), ys.ravel()]

In [ ]:
response = _check_boundary_response_method(nn, "auto")(
    X_grid
)  # the function returns the probability of points(inputs) belong to each class

if len(response.shape) != 1:
    response = response[
        :, 1
    ]  # since there's only 2 classes so 1 can represent the other(the prob sums up to 1)


cm = plt.cm.RdBu
display = DecisionBoundaryDisplay(
    xx0=xs, xx1=ys, response=response.reshape(xs.shape)
)  # class to draw DecisionBoundary
display.plot(cmap=cm, alpha=0.8)
plt.show()

In [ ]:
itr = 20

In [ ]:
nn.device

In [ ]:
np.array([[10.2, -16.5]])

In [ ]:
import contextlib
import copy
import random

import numpy as np
import torch
from sklearn.neighbors import KDTree


def get_tracin(X, label, nn, iter=1, train=None, mode="random", num=None, resample=False):
    loss1 = []
    loss2 = []
    tracin = []
    if num is None:
        if mode == "neighbor":
            num = 10
        elif mode in ("random", "uniform"):
            num = int(len(X) / 4)
    if not isinstance(label, np.ndarray):
        label = np.array([label for i in range(len(X))])
    if train is None:
        for i in range(len(X)):
            x = torch.tensor(X[i]).to(torch.float32)
            label_t = torch.tensor(label[i]).to(torch.float32)
            output1 = nn.model(x)
            loss1.append(nn.criterion(output1.squeeze(-1), label_t).detach().cpu().numpy())
            nn_next = copy.deepcopy(nn)
            for j in range(iter):
                nn_next.train_epoch(x, label_t)
            output2 = nn_next.model(x.to(device=nn.device))
            loss2.append(nn_next.criterion(output2.squeeze(-1).to(device=nn.device), label_t.to(device=nn.device)).detach().cpu().numpy())
            del nn_next
            tracin.append(loss1[-1] - loss2[-1])
    else:
        for i in range(len(X)):
            output1 = nn.model(torch.tensor(X[i]).to(torch.float32).to(device=nn.device))
            loss1.append(nn.criterion(output1.squeeze(-1), torch.tensor(label[i]).to(torch.float32).to(device=nn.device)).detach().cpu().numpy())
            nn_next = copy.deepcopy(nn)
            for j in range(iter):
                if mode == "neighbor":
                    tree = KDTree(train[0])
                    dist, ind = tree.query(X[[i]], k=num)
                    x = np.append(train[0][ind[0]], X[[i]], axis=0)
                    x = torch.from_numpy(x).to(dtype=torch.float32)
                    label_t = np.append(train[1][ind[0]], label[i])
                    label_t = torch.from_numpy(label_t).to(dtype=torch.float32)
                elif mode == "random":
                    if resample:
                        random.seed(j)
                    else:
                        random.seed(1)
                    ind = random.sample(range(len(train[0])), num)
                    x = np.append(train[0][ind], X[[i]], axis=0)
                    x = torch.from_numpy(x).to(dtype=torch.float32)
                    label_t = np.append(train[1][ind], label[i])
                    label_t = torch.from_numpy(label_t).to(dtype=torch.float32)
                elif mode == "uniform":
                    x_min, x_max = train[0][:, 0].min(), train[0][:, 0].max()
                    y_min, y_max = train[0][:, 1].min(), train[0][:, 1].max()
                    grid_size = (x_max - x_min) / int(num**0.5)
                    grid = {}
                    # Assign points to grid cells
                    for idx, point in enumerate(train[0]):
                        x_idx = int((point[0] - x_min) / grid_size)
                        y_idx = int((point[1] - y_min) / grid_size)
                        grid_cell = (x_idx, y_idx)
                        if grid_cell not in grid:
                            grid[grid_cell] = []
                        grid[grid_cell].append((point, idx))
                    # Select one point per occupied grid cell and get their indices
                    if resample:
                        ind = []
                        for pts in grid.values():
                            for k in range(j):
                                with contextlib.suppress(Exception):
                                    idx = pts[j-k][1]
                            ind.append(idx)
                    else:
                        ind = [pts[0][1] for pts in grid.values()]
                    x = torch.from_numpy(train[0][ind]).to(dtype=torch.float32)
                    label_t = torch.from_numpy(train[1][ind]).to(dtype=torch.float32)
                if j ==0:
                    plt.scatter(x[:,0], x[:,1], c=label_t, cmap=ListedColormap(["#FF0000", "#0000FF"]))
                    plt.show()
                nn_next.train_epoch(x, label_t)
                response = _check_boundary_response_method(nn_next, "auto")(
                    X_grid
                )  # the function returns the probability of points(inputs) belong to each class

                if len(response.shape) != 1:
                    response = response[
                        :, 1
                    ]  # since there's only 2 classes so 1 can represent the other(the prob sums up to 1)


                cm = plt.cm.RdBu
                display = DecisionBoundaryDisplay(
                    xx0=xs, xx1=ys, response=response.reshape(xs.shape)
                )  # class to draw DecisionBoundary
                display.plot(cmap=cm, alpha=0.8)
                plt.title(f"{j}")
                plt.show()
                #plt.savefig(f"iter={j}")
            output2 = nn_next.model(torch.tensor(X[i]).to(torch.float32).to(device=nn.device))
            loss2.append(nn_next.criterion(output2.squeeze(-1).to(device=nn.device), torch.tensor(label[i]).to(torch.float32).to(device=nn.device)).detach().cpu().numpy())
            del nn_next
            tracin.append(loss1[-1] - loss2[-1])
    return np.array(loss1), np.array(loss2), np.array(tracin)


In [ ]:
loss0_1, loss0_2, tracin_0 = get_tracin(np.array([[10.2, -16.5]]), 0., nn, train=(X_train, y_train), iter=80, mode="random", num=500, resample=resample)
#loss1_1, loss1_2, tracin_1 = get_tracin(X_grid, 1., nn, train=(X_train, y_train), iter=itr, mode="random", num=500, resample=resample)
#tracin = tracin_0 + tracin_1

In [ ]:
from ppu.methods.tracin import get_tracin

loss0_1, loss0_2, tracin_0 = get_tracin(X_grid, 0., nn, train=(X_train, y_train), iter=3, mode="random", num=500, resample=resample)
loss1_1, loss1_2, tracin_1 = get_tracin(X_grid, 1., nn, train=(X_train, y_train), iter=3, mode="random", num=500, resample=resample)
tracin = tracin_0 + tracin_1

In [ ]:

losso_t_1, losso_t_2, tracino_t = get_tracin(X_train, 1., nn, train=(X_train, y_train), iter=itr, mode="random", num=500, resample=resample)
lossi_t_1, lossi_t_2, tracini_t = get_tracin(X_train, 0., nn, train=(X_train, y_train), iter=itr, mode="random", num=500, resample=resample)
tracin_t = tracino_t + tracini_t

In [ ]:
tracin_t.max()

In [ ]:
tracin_0 = tracin_0.reshape(xs.shape)
tracin_1 = tracin_1.reshape(xs.shape)
tracin = tracin.reshape(xs.shape)

In [ ]:
figure, axs = plt.subplots(2, 3, figsize=(15, 7))
cm_bright = ListedColormap(["#FF0000", "#0000FF"])  # color for data points
ax = axs[0][0]  # position of subgraph
ax.set_title("Data")  # subgraph title
# Plot the training points
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright, edgecolors="k")
ax.set_xlim(x_min, x_max)  # axis range
ax.set_ylim(y_min, y_max)

ax = axs[0][1]
ax.hist(tracin, bins="auto", log=True)
ax = axs[0][2]
ax.hist(tracin_t, bins="auto", log=True)


ax = axs[1][0]
draw(tracin_0, x, y, ax=ax)
ax = axs[1][1]
draw(tracin_1, x, y, ax=ax)
ax = axs[1][2]
draw(tracin, x, y, ax=ax)



plt.subplots_adjust(left=0.12,
        bottom=0.1,
        right=0.85,
        top=0.9,
        wspace=0.54,
        hspace=0.2,
    )


In [ ]:
from ppu.viz.tracin_plot import tracin_plot_single

figure = tracin_plot_single(RingBlobs, 2000, classifier, 2, seed=0, itr=3)

In [ ]:
nn.optimizer

In [ ]:
tracin_0 = abs(tracin_0.reshape(xs.shape))
tracin_1 = abs(tracin_1.reshape(xs.shape))
tracin = tracin_0 + tracin_1
tracin = tracin.reshape(xs.shape)

In [ ]:
figure, axs = plt.subplots(2, 3, figsize=(15, 7))
cm_bright = ListedColormap(["#FF0000", "#0000FF"])  # color for data points
ax = axs[0][0]  # position of subgraph
ax.set_title("Data")  # subgraph title
# Plot the training points
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright, edgecolors="k")
ax.set_xlim(x_min, x_max)  # axis range
ax.set_ylim(y_min, y_max)

ax = axs[0][1]
ax.axis("off")
ax = axs[0][2]
ax.axis("off")


ax = axs[1][0]
draw(tracin_0, x, y, ax=ax)
ax = axs[1][1]
draw(tracin_1, x, y, ax=ax)
ax = axs[1][2]
draw(tracin, x, y, ax=ax)



plt.subplots_adjust(left=0.12,
        bottom=0.1,
        right=0.85,
        top=0.9,
        wspace=0.54,
        hspace=0.2,
    )


In [ ]:
from ppu.viz.tracin_plot import tracin_plot_seeds

seeds = list(range(4))
figure = tracin_plot_seeds(RingBlobs, 2000, classifier, seeds)

In [ ]:
figure, axs = plt.subplots(4, 3, figsize=(15, 18))

gens = [Circular, GaussianBlobs, Moons, RingBlobs]
for i in range(len(gens)):
    gen = gens[i]
    seed = 2

    data = get_dataset(seed, gen, n_samples=n_samples)
    nn  = get_models(classifier, gen, reps=1, n_samples=n_samples, seeds=[seed])[0]

    (X_train, y_train), (X_test, y_test) = data
    eps = 1.
    x_min, x_max = X_train[:, 0].min() - eps, X_train[:, 0].max() + eps
    y_min, y_max = X_train[:, 1].min() - eps, X_train[:, 1].max() + eps

    x = np.linspace(
            x_min, x_max, n_ticks
        )  # for the heatmap we need to creat the grid first, so n_ticks are the density of the grid nodes
    y = np.linspace(y_min, y_max, n_ticks)
    xs, ys = np.meshgrid(x, y)
    X_grid = np.c_[xs.ravel(), ys.ravel()]


    cm_bright = ListedColormap(["#FF0000", "#0000FF"])  # color for data points
    ax = axs[i][0]  # position of subgraph
    ax.set_title("Data")  # subgraph title
    # Plot the training points
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright, edgecolors="k")
    ax.set_xlim(x_min, x_max)  # axis range
    ax.set_ylim(y_min, y_max)


    loss0_1, loss0_2, tracin_0 = get_tracin(X_grid, 0., nn, train=(X_train, y_train), iter=3, mode="random", num=500)
    loss1_1, loss1_2, tracin_1 = get_tracin(X_grid, 1., nn, train=(X_train, y_train), iter=3, mode="random", num=500)
    tracin = tracin_0 + tracin_1

    tracin_0 = tracin_0.reshape(xs.shape)
    tracin_1 = tracin_1.reshape(xs.shape)
    tracin = tracin.reshape(xs.shape)

    ax = axs[i][1]
    draw(tracin, x, y, ax=ax)


    loss0_1, loss0_2, tracin_0 = get_tracin(X_grid, 0., nn, train=(X_train, y_train), iter=3, mode="random", num=500, resample=True)
    loss1_1, loss1_2, tracin_1 = get_tracin(X_grid, 1., nn, train=(X_train, y_train), iter=3, mode="random", num=500, resample=True)
    tracin = tracin_0 + tracin_1

    tracin_0 = tracin_0.reshape(xs.shape)
    tracin_1 = tracin_1.reshape(xs.shape)
    tracin = tracin.reshape(xs.shape)

    ax = axs[i][2]
    draw(tracin, x, y, ax=ax)

    print(i)


plt.subplots_adjust(left=0.11,
        bottom=0.12,
        right=0.85,
        top=0.88,
        wspace=0.45,
        hspace=0.3,
    )

In [ ]:
from ppu.viz.tracin_plot import tracin_plot_iter

iters = [i+1 for i in range(5)]
figure = tracin_plot_iter(Moons, 2000, classifier, iters)

In [ ]:
def get_prscore(X, label, nn, iter=1, train=None, nbr=10):
    score1 = []
    score2 = []
    score_diff = []
    if not isinstance(label, np.ndarray):
        label = np.array([label for i in range(len(X))])
    if train is None:
        for i in range(len(X)):
            x = torch.tensor(X[i]).to(torch.float32)
            score1.append(nn.predict_proba(x.numpy()))
            nn_next = copy.deepcopy(nn)
            for i in range(iter):
                nn_next.train_epoch(x, torch.tensor(label))
            score2.append(nn_next.predict_proba(x.numpy()))
            del nn_next
            score_diff.append(abs(score1[-1] - score2[-1]))
    else:
        for i in range(len(X)):
            tree = KDTree(train[0])
            dist, ind = tree.query(X[[i]], k=nbr)
            x = np.append(train[0][ind[0]], X[[i]], axis=0)
            x = torch.from_numpy(x).to(dtype=torch.float32)
            label_t = np.append(train[1][ind[0]], label[i])
            label_t = torch.from_numpy(label_t).to(dtype=torch.float32)
            score1.append(nn.predict_proba(X[i]))
            nn_next = copy.deepcopy(nn)
            for j in range(iter):
                nn_next.train_epoch(x, label_t)
            score2.append(nn_next.predict_proba(X[i]))
            del nn_next
            score_diff.append(score1[-1] - score2[-1])
    return np.array(score1), np.array(score2), np.array(score_diff)

In [ ]:
score0_1, score0_2, score_diff_0 = get_prscore(X_grid, 0., nn, train=(X_train, y_train))

In [ ]:
score1_1, score1_2, score_diff_1 = get_prscore(X_grid, 1., nn, train=(X_train, y_train))

In [ ]:
score_diff = score_diff_0 + score_diff_1

In [ ]:
score_diff_0 = score_diff_0.reshape(xs.shape)
score_diff_1 = score_diff_1.reshape(xs.shape)
score_diff = score_diff.reshape(xs.shape)

In [ ]:
figure, axs = plt.subplots(2, 3, figsize=(15, 7.5))
cm_bright = ListedColormap(["#FF0000", "#0000FF"])  # color for data points
ax = axs[0][0]  # position of subgraph
ax.set_title("Data")  # subgraph title
# Plot the training points
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright, edgecolors="k")
ax.set_xlim(x_min, x_max)  # axis range
ax.set_ylim(y_min, y_max)
ax.set_xticks(())
ax.set_yticks(())

ax = axs[0][1]
ax.axis("off")
ax = axs[0][2]
ax.axis("off")


ax = axs[1][0]
draw(score_diff_0, x, y, ax=ax)
ax = axs[1][1]
draw(score_diff_1, x, y, ax=ax)
ax = axs[1][2]
draw(score_diff, x, y, ax=ax)



plt.subplots_adjust(left=0.12,
        bottom=0.1,
        right=0.85,
        top=0.9,
        wspace=0.54,
        hspace=0.1,
    )


In [ ]:
draw(score_diff_0, x, y)

In [ ]:
draw(score_diff_1, x, y)

In [ ]:
draw(score_diff, x, y)

In [ ]:
output1 = nn.model(test_point)
loss1 = nn.criterion(output1.squeeze(-1), torch.tensor(test_point_label))
nn_next = copy.deepcopy(nn)
for i in range(1):
    nn_next.train_epoch(test_point, torch.tensor(test_point_label))
output2 = nn_next.model(test_point)
loss2 = nn_next.criterion(output2.squeeze(-1), torch.tensor(test_point_label))

In [ ]:
test_point = torch.tensor([1., -1.]).to(torch.float32)
test_point_label = 0.

In [ ]:
output1 = nn.model(test_point)
loss1 = nn.criterion(output1.squeeze(-1), torch.tensor(test_point_label))
nn_next = copy.deepcopy(nn)
for i in range(3):
    nn_next.train_epoch(test_point, torch.tensor(test_point_label))
output2 = nn_next.model(test_point)
loss2 = nn_next.criterion(output2.squeeze(-1), torch.tensor(test_point_label))


print(nn_next.predict_proba(np.array(test_point)))

response = _check_boundary_response_method(nn_next, "auto")(
    X_grid
)  # the function returns the probability of points(inputs) belong to each class

if len(response.shape) != 1:
    response = response[
        :, 1
    ]  # since there's only 2 classes so 1 can represent the other(the prob sums up to 1)

display = DecisionBoundaryDisplay(
    xx0=xs, xx1=ys, response=response.reshape(xs.shape)
)  # class to draw DecisionBoundary
display.plot(cmap=cm, alpha=0.8)

del nn_next
tracin_0 = loss1 - loss2
tracin_0

In [ ]:
test_point_label = 1.

In [ ]:
output1 = nn.model(test_point)
loss1 = nn.criterion(output1.squeeze(-1), torch.tensor(test_point_label))
nn_next = copy.deepcopy(nn)
nn_next.train_epoch(test_point, torch.tensor(test_point_label))
output2 = nn_next.model(test_point)
loss2 = nn_next.criterion(output2.squeeze(-1), torch.tensor(test_point_label))

response = _check_boundary_response_method(nn_next, "auto")(
    X_grid
)  # the function returns the probability of points(inputs) belong to each class

if len(response.shape) != 1:
    response = response[
        :, 1
    ]  # since there's only 2 classes so 1 can represent the other(the prob sums up to 1)

display = DecisionBoundaryDisplay(
    xx0=xs, xx1=ys, response=response.reshape(xs.shape)
)  # class to draw DecisionBoundary
display.plot(cmap=cm, alpha=0.8)

del nn_next
tracin_1 = loss1 - loss2
tracin_1

In [ ]:
bu = tracin_0 + tracin_1
bu

In [ ]:
from copy import deepcopy

In [ ]:
test_point = torch.tensor([1.5, -1.]).to(torch.float32)

In [ ]:
nn  = get_models(classifier, gen, reps=1, n_samples=n_samples, seeds=[seed])[0]

In [ ]:
output1 = nn.model(test_point)
loss1 = nn.criterion(output1.squeeze(-1), torch.tensor(0.))
nn_next = copy.deepcopy(nn)
nn_next.train_epoch(test_point, torch.tensor(0.))
output2 = nn_next.model(test_point)
loss2 = nn_next.criterion(output2.squeeze(-1), torch.tensor(0.))
tracin__0 = loss1 - loss2

In [ ]:
(X_train, y_train), (X_test, y_test) = get_dataset(seed, gen, n_samples=n_samples)
X_train = np.append(X_train, test_point.numpy().reshape((1,2)), axis=0)
y_train = np.append(y_train, 0)
new_clf = deepcopy(classifier)
new_clf.fit(X_train, y_train)

In [ ]:
noutput1 = new_clf.model(test_point)
nloss1 = new_clf.criterion(noutput1.squeeze(-1), torch.tensor(0.))
nn_next = copy.deepcopy(new_clf)
nn_next.train_epoch(test_point, torch.tensor(0.))
noutput2 = nn_next.model(test_point)
nloss2 = nn_next.criterion(noutput2.squeeze(-1), torch.tensor(0.))
ntracin__0 = nloss1 - nloss2

In [ ]:
loss1

In [ ]:
nloss1

In [ ]:
loss2

In [ ]:
nloss2